# 05 - Equity Data Quality Analysis

Analyze demographic data completeness, compute equity disparity indices, and generate composite quality scores. This is the most policy-relevant notebook — incomplete demographic data directly undermines equity analysis.

**Toolkit modules used**: `cj_data_quality.coverage`, `cj_data_quality.validation`, `cj_data_quality.visualization`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from cj_data_quality.notebook_utils import setup_notebook, display_quality_score
from cj_data_quality.sample_data import generate_and_load
from cj_data_quality.coverage.equity_coverage import (
    analyze_demographic_completeness,
    compute_equity_disparity_index,
)
from cj_data_quality.validation.completeness_scorer import (
    score_completeness,
    score_consistency,
    score_validity,
    score_uniqueness,
    compute_composite_score,
)
from cj_data_quality.visualization.heatmaps import plot_equity_heatmap
from cj_data_quality.visualization.plots import plot_quality_scorecard

setup_notebook()
print("Setup complete.")

## Load Data

In [ ]:
df = generate_and_load(n_records=50000, seed=42)
print(f"Shape: {df.shape}")
print(f"States: {df['state_code'].nunique()}")

## Demographic Completeness Analysis

How complete are race, ethnicity, sex, and age fields across states?

In [ ]:
demo_coverage = analyze_demographic_completeness(df, state_col="state_code")
print(f"Coverage results: {len(demo_coverage)}")

# Show worst completeness
demo_df = pd.DataFrame([
    {
        "State": ec.state_code,
        "Field": ec.field_name,
        "Completeness": ec.completeness,
        "Distinct Values": ec.distinct_values,
        "Most Common": ec.most_common_value,
    }
    for ec in demo_coverage
])

worst = demo_df.nsmallest(15, "Completeness")
worst["Completeness"] = worst["Completeness"].apply(lambda x: f"{x:.1%}")
worst

## Equity Coverage Heatmap

In [ ]:
equity_viz = pd.DataFrame([
    {"state_code": ec.state_code, "field_name": ec.field_name, "completeness": ec.completeness}
    for ec in demo_coverage
])

# Show top 20 states by worst average demographic completeness
state_avg = equity_viz.groupby("state_code")["completeness"].mean().nsmallest(20).index
equity_subset = equity_viz[equity_viz["state_code"].isin(state_avg)]

fig = plot_equity_heatmap(
    equity_subset,
    title="Demographic Data Completeness: 20 Worst States",
    figsize=(10, 10),
)
plt.show()

## Equity Disparity Index

Measure how evenly population metrics are distributed across racial groups. Higher coefficient of variation = more disparity in group representation.

In [ ]:
# Filter to records with race data
with_race = df.dropna(subset=["race", "total_population"])

disparity = compute_equity_disparity_index(
    with_race, state_col="state_code", group_col="race", metric_col="total_population"
)

disparity_df = pd.DataFrame([
    {"State": k, "Disparity Index (CV)": v}
    for k, v in sorted(disparity.items(), key=lambda x: x[1], reverse=True)
])

print("Top 15 states by racial disparity in population counts:")
disparity_df.head(15)

## Composite Quality Scoring

Compute a weighted quality score across 5 dimensions: completeness, consistency, timeliness, validity, uniqueness.

In [ ]:
# Overall dataset quality
overall_score = compute_composite_score(df, entity_name="Corrections Dataset (All States)")
score_df = display_quality_score(overall_score)
score_df

In [ ]:
fig = plot_quality_scorecard(overall_score)
plt.show()

## Per-State Quality Scores

In [ ]:
# Score a few representative states
states_to_score = ["US_CA", "US_TX", "US_NY", "US_AK", "US_WY"]

state_scores = []
for state in states_to_score:
    state_df = df[df["state_code"] == state]
    score = compute_composite_score(state_df, entity_name=state)
    state_scores.append({
        "State": state,
        "Composite Score": f"{score.composite_score:.2f}",
        "Grade": score.grade,
        **{ds.dimension.value.title(): f"{ds.score:.2f}" for ds in score.dimension_scores}
    })

pd.DataFrame(state_scores)

## Key Findings

1. **Demographic completeness varies dramatically** — some states have >95% race/ethnicity data, others <60%
2. **Ethnicity** is consistently the worst-reported demographic field across states
3. **Equity implications**: States with poor demographic data cannot support racial disparity analysis, which is core to Recidiviz's mission
4. **Composite scoring** provides a single actionable number per state for prioritizing data quality improvement efforts
5. **The disparity index** highlights where group-level representation may be skewed by data quality issues vs. actual population patterns

This analysis directly supports Recidiviz's stated mission: *"Every person deserves a fair chance."* Fair analysis requires complete, equitable data.